# 04 — DistilBERT Fine-Tuning

## Theory

### What is Pretraining?
Pretraining means training a large model on a massive corpus (e.g., all of Wikipedia
and BookCorpus) using a self-supervised task — for BERT, this is **Masked Language
Modeling** (MLM): randomly mask 15% of tokens and train the model to predict them.
This forces the model to learn deep representations of language structure, meaning,
and world knowledge — without any labeled data.

### Why Does Fine-Tuning Work?
The pretrained model has already learned:
- Syntax and grammar (from seeing billions of sentences)
- Semantic relationships ("fraud" ≈ "scam" ≈ "unauthorized")
- Domain-general reasoning patterns

Fine-tuning adds a small classification head and updates all parameters on labeled
complaint data. Because the backbone is already powerful, only a few epochs on
a small dataset are needed — this is **transfer learning**.

### DistilBERT vs BERT
DistilBERT is a **distilled** (compressed) version of BERT-base: 40% fewer parameters,
60% faster, retains 97% of BERT's performance. Ideal for limited-compute scenarios.

### Why Transformers Are Superior
Unlike RNNs, Transformers use **self-attention** that connects every token to every
other token in O(1) steps — capturing long-range dependencies perfectly. Combined
with pretraining, they consistently outperform all prior architectures on NLP tasks.

Prerequisites: `python src/data_prep.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
from pathlib import Path

from src.transformer_model import train_transformer, predict_transformer, load_transformer
from src.data_prep import compute_class_weights
from src.evaluate import evaluate_model, plot_confusion_matrix

sns.set_theme(style='whitegrid')
BASE_DIR   = Path('..')
DATA_DIR   = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

le_product = joblib.load(MODELS_DIR / 'label_encoder_product.joblib')
label_names = list(le_product.classes_)
NUM_CLASSES = len(label_names)

# Use raw (uncleaned) narrative for DistilBERT — it handles casing, punctuation natively
train_texts  = train_df['Consumer complaint narrative'].fillna('').tolist()
train_labels = train_df['product_encoded'].tolist()
val_texts    = val_df['Consumer complaint narrative'].fillna('').tolist()
val_labels   = val_df['product_encoded'].tolist()
test_texts   = test_df['Consumer complaint narrative'].fillna('').tolist()
test_labels  = test_df['product_encoded'].tolist()

print(f'Classes: {NUM_CLASSES} | Train: {len(train_texts):,}')

## 2. Class Weights

In [ ]:
class_weights = compute_class_weights(np.array(train_labels))
print('Class weights:')
for name, w in zip(label_names, class_weights):
    print(f'  {name[:40]:<40}: {w:.3f}')

## 3. Fine-Tune DistilBERT — Primary Task (Product)

In [ ]:
model, tokenizer, trainer = train_transformer(
    train_texts=train_texts,
    train_labels=train_labels,
    val_texts=val_texts,
    val_labels=val_labels,
    num_classes=NUM_CLASSES,
    label_names=label_names,
    class_weights=class_weights,
    max_length=256,
    batch_size=16,
    learning_rate=2e-5,
    num_epochs=3,
    task='product',
)
print('Fine-tuning complete.')

## 4. Evaluate on Test Set

In [ ]:
test_preds, test_probas = predict_transformer(
    model, tokenizer, test_texts,
    max_length=256, batch_size=32, device=DEVICE,
)

metrics = evaluate_model(
    np.array(test_labels), test_preds, test_probas,
    label_names=label_names,
    model_name='distilbert_product',
)

print(f"Test Results:")
print(f"  Accuracy:    {metrics['accuracy']:.4f}")
print(f"  Macro F1:    {metrics['f1_macro']:.4f}")
print(f"  Weighted F1: {metrics['f1_weighted']:.4f}")
print(f"\n{metrics['classification_report']}")

In [ ]:
plot_confusion_matrix(
    np.array(test_labels), test_preds,
    label_names=label_names,
    model_name='distilbert_product',
)
from IPython.display import Image
Image(str(RESULTS_DIR / 'confusion_matrix_distilbert_product.png'))

## 5. Fine-Tune for Secondary Task (Issue Group)

In [ ]:
le_issue = joblib.load(MODELS_DIR / 'label_encoder_issue_group.joblib')
issue_label_names = list(le_issue.classes_)

train_issue_labels = train_df['issue_group_encoded'].tolist()
val_issue_labels   = val_df['issue_group_encoded'].tolist()
test_issue_labels  = test_df['issue_group_encoded'].tolist()

issue_weights = compute_class_weights(np.array(train_issue_labels))

model_iss, tokenizer_iss, trainer_iss = train_transformer(
    train_texts=train_texts,
    train_labels=train_issue_labels,
    val_texts=val_texts,
    val_labels=val_issue_labels,
    num_classes=len(issue_label_names),
    label_names=issue_label_names,
    class_weights=issue_weights,
    max_length=256,
    batch_size=16,
    learning_rate=2e-5,
    num_epochs=3,
    task='issue_group',
)

iss_preds, iss_probas = predict_transformer(
    model_iss, tokenizer_iss, test_texts,
    max_length=256, batch_size=32, device=DEVICE,
)

iss_metrics = evaluate_model(
    np.array(test_issue_labels), iss_preds, iss_probas,
    label_names=issue_label_names,
    model_name='distilbert_issue_group',
)

print(f"Issue Group Test Results:")
print(f"  Accuracy: {iss_metrics['accuracy']:.4f} | Macro F1: {iss_metrics['f1_macro']:.4f}")

## 6. Training Logs

In [ ]:
# Plot trainer log history
log_history = trainer.state.log_history
train_logs = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if train_logs:
    steps = [x['step'] for x in train_logs]
    losses = [x['loss'] for x in train_logs]
    axes[0].plot(steps, losses, color='steelblue')
    axes[0].set_title('Training Loss', fontweight='bold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)

if eval_logs:
    steps = [x['step'] for x in eval_logs]
    f1s   = [x.get('eval_f1_macro', 0) for x in eval_logs]
    axes[1].plot(steps, f1s, color='green', marker='o', markersize=4)
    axes[1].set_title('Validation Macro F1', fontweight='bold')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(alpha=0.3)

plt.suptitle('DistilBERT Fine-Tuning Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'distilbert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

DistilBERT fine-tuning leverages:
1. **Pretrained contextual representations** — tokens are understood in context, not isolation.
2. **WordPiece tokenization** — handles OOV words, financial jargon, and misspellings.
3. **Bidirectional self-attention** — every token attends to every other token simultaneously.

Models saved to `models/distilbert_best_product/` and `models/distilbert_best_issue_group/`.
Results saved to `results/distilbert_product_metrics.json`.